In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import os

drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Store-Sales-Forecasting"
PROCESSED_PATH = f"{PROJECT_PATH}/Data/Processed"

master_df = pd.read_csv(
    f"{PROCESSED_PATH}/master_dataset.csv",
    parse_dates=["date"]
)

print("Master dataset loaded.")
print("Shape:", master_df.shape)
master_df.head()

Mounted at /content/drive
Master dataset loaded.
Shape: (3000888, 18)


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,holiday_type,holiday_locale,holiday_description,is_transferred,is_holiday,transactions_missing
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,0.0,93.14,Holiday,National,Primer dia del ano,False,1,1
1,1194,2013-01-01,42,CELEBRATION,0.0,0,Cuenca,Azuay,D,2,0.0,93.14,Holiday,National,Primer dia del ano,False,1,1
2,1193,2013-01-01,42,BREAD/BAKERY,0.0,0,Cuenca,Azuay,D,2,0.0,93.14,Holiday,National,Primer dia del ano,False,1,1
3,1192,2013-01-01,42,BOOKS,0.0,0,Cuenca,Azuay,D,2,0.0,93.14,Holiday,National,Primer dia del ano,False,1,1
4,1191,2013-01-01,42,BEVERAGES,0.0,0,Cuenca,Azuay,D,2,0.0,93.14,Holiday,National,Primer dia del ano,False,1,1


In [2]:
df = master_df.copy()

df = df.sort_values(
    by=["store_nbr", "family", "date"]
).reset_index(drop=True)

print(df.shape)

(3000888, 18)


In [3]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week_num"] = df["date"].dt.dayofweek

# Monday=0 ... Sunday=6
df["is_weekend"] = df["day_of_week_num"].isin([5, 6]).astype(int)

# Quarter: 1, 2, 3, or 4
df["quarter"] = df["date"].dt.quarter

print(
    df[
        ["date", "year", "month", "day",
         "day_of_week_num", "is_weekend", "quarter"]
    ].head()
)

        date  year  month  day  day_of_week_num  is_weekend  quarter
0 2013-01-01  2013      1    1                1           0        1
1 2013-01-02  2013      1    2                2           0        1
2 2013-01-03  2013      1    3                3           0        1
3 2013-01-04  2013      1    4                4           0        1
4 2013-01-05  2013      1    5                5           1        1


In [4]:
# Whether at least one item is on promotion
df["has_promotion"] = (df["onpromotion"] > 0).astype(int)

# Log transform reduces the effect of very large promotion counts
df["log_onpromotion"] = np.log1p(df["onpromotion"])

df[["onpromotion", "has_promotion", "log_onpromotion"]].head()

,onpromotion,has_promotion,log_onpromotion
0,0,0,0.0
1,0,0,0.0
2,0,0,0.0
3,0,0,0.0
4,0,0,0.0


In [5]:
group_cols = ["store_nbr", "family"]

df["sales_lag_1"] = df.groupby(group_cols)["sales"].shift(1)
df["sales_lag_7"] = df.groupby(group_cols)["sales"].shift(7)
df["sales_lag_14"] = df.groupby(group_cols)["sales"].shift(14)

In [6]:
df["sales_rolling_mean_7"] = (
    df.groupby(group_cols)["sales"]
    .transform(lambda x: x.shift(1).rolling(window=7).mean())
)

df["sales_rolling_mean_14"] = (
    df.groupby(group_cols)["sales"]
    .transform(lambda x: x.shift(1).rolling(window=14).mean())
)

In [7]:
lag_columns = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_rolling_mean_7",
    "sales_rolling_mean_14"
]

print(df[lag_columns].isnull().sum())

sales_lag_1               1782
sales_lag_7              12474
sales_lag_14             24948
sales_rolling_mean_7     12474
sales_rolling_mean_14    24948
dtype: int64


In [8]:
df_model = df.dropna(subset=lag_columns).copy()

print("Before removing early lag rows:", df.shape)
print("After removing early lag rows:", df_model.shape)

Before removing early lag rows: (3000888, 31)
After removing early lag rows: (2975940, 31)


In [9]:
df_model.to_csv(
    f"{PROCESSED_PATH}/model_ready_dataset.csv",
    index=False
)

print("Model-ready dataset saved successfully.")

Model-ready dataset saved successfully.


In [10]:
lag_columns = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_rolling_mean_7",
    "sales_rolling_mean_14"
]

print(df[lag_columns].isnull().sum())

sales_lag_1               1782
sales_lag_7              12474
sales_lag_14             24948
sales_rolling_mean_7     12474
sales_rolling_mean_14    24948
dtype: int64


In [11]:
df_model = df.dropna(subset=lag_columns).copy()

print("Before removing early lag rows:", df.shape)
print("After removing early lag rows:", df_model.shape)

Before removing early lag rows: (3000888, 31)
After removing early lag rows: (2975940, 31)


In [12]:
feature_columns = [
    "year", "month", "day", "day_of_week_num",
    "is_weekend", "quarter",
    "has_promotion", "log_onpromotion",
    "sales_lag_1", "sales_lag_7", "sales_lag_14",
    "sales_rolling_mean_7", "sales_rolling_mean_14"
]

print(df_model[feature_columns].head())
print("\nFinal model dataset shape:", df_model.shape)

    year  month  day  day_of_week_num  is_weekend  quarter  has_promotion  \
14  2013      1   15                1           0        1              0   
15  2013      1   16                2           0        1              0   
16  2013      1   17                3           0        1              0   
17  2013      1   18                4           0        1              0   
18  2013      1   19                5           1        1              0   

    log_onpromotion  sales_lag_1  sales_lag_7  sales_lag_14  \
14              0.0          2.0          2.0           0.0   
15              0.0          1.0          2.0           2.0   
16              0.0          1.0          2.0           3.0   
17              0.0          1.0          3.0           3.0   
18              0.0          0.0          2.0           5.0   

    sales_rolling_mean_7  sales_rolling_mean_14  
14              2.142857               2.142857  
15              2.000000               2.214286  
16      

In [13]:
df_model.to_csv(
    f"{PROCESSED_PATH}/model_ready_dataset.csv",
    index=False
)

print("Model-ready dataset saved successfully.")

Model-ready dataset saved successfully.
